# Bellman Equations Programming Assignment

### Mukul Kumar Dhali
### University of Cumberland, School of Computer and Information Sciences
### PhDAI 830: Applied Machine Intelligence and Reinforcement Learning
### Professor Dr. Soamar Homsi​
### September 06, 2026


**Project description:** This notebook implements the Bellman Expectation Equation and Bellman Optimality Equation for Markov Decision Processes (MDPs).




## 1. Setup MDP Components
States, actions, transition probabilities, and rewards.

In [7]:
import numpy as np

n_states = 3
n_actions = 2
gamma = 0.9

P = np.array([
    [
        [0.8, 0.2, 0.0],
        [0.0, 1.0, 0.0]
    ],
    [
        [0.0, 0.5, 0.5],
        [0.0, 0.0, 1.0]
    ],
    [
        [0.0, 0.0, 1.0],
        [0.0, 0.0, 1.0]
    ]
])

R = np.array([
    [
        [1.0, 0.0, 0.0],
        [0.0, 2.0, 0.0]
    ],
    [
        [0.0, 1.0, 3.0],
        [0.0, 0.0, 4.0]
    ],
    [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0]
    ]
])

pi = np.array([
    [0.5, 0.5],
    [1.0, 0.0],
    [0.0, 1.0]
])

print("MDP Loaded.")

MDP Loaded.


## 2. Bellman Expectation Equation for V

In [8]:
def bellman_expectation_V(P, R, pi, gamma, V):
    n_states, n_actions, _ = P.shape
    V_new = np.zeros_like(V)
    for s in range(n_states):
        value = 0.0
        for a in range(n_actions):
            for s_next in range(n_states):
                value += pi[s, a] * P[s, a, s_next] * (R[s, a, s_next] + gamma * V[s_next])
        V_new[s] = value
    return V_new

V = np.zeros(n_states)
for i in range(10):
    V = bellman_expectation_V(P, R, pi, gamma, V)
    print(f"Iteration {i}: V = {V}")

Iteration 0: V = [1.4 2.  0. ]
Iteration 1: V = [2.984 2.9   0.   ]
Iteration 2: V = [4.04024 3.305   0.     ]
Iteration 3: V = [4.6391864 3.48725   0.       ]
Iteration 4: V = [4.9532221 3.5692625 0.       ]
Iteration 5: V = [5.11056171 3.60616812 0.        ]
Iteration 6: V = [5.187133   3.62277566 0.        ]
Iteration 7: V = [5.22366674 3.63024905 0.        ]
Iteration 8: V = [5.24085451 3.63361207 0.        ]
Iteration 9: V = [5.24885814 3.63512543 0.        ]


### Interpretation of State-Value Function Results

The values in \(V^\pi\) represent the expected long-term return when starting in each state and following the policy \(\pi\). Several patterns appear in the iterations:

- The values increase over time because each update incorporates additional discounted future rewards.
- Convergence occurs as the updates stabilize, indicating that the recursive Bellman expectation equation has reached a fixed point.
- State 0 has the highest value because its transitions lead to higher immediate rewards and beneficial future states.
- State 2 remains at zero because it transitions only to itself with zero reward, making it an absorbing state with no future return.

These results demonstrate how the Bellman expectation equation captures both immediate and future rewards under a fixed policy.


## 3. Bellman Expectation Equation for Q

In [9]:
def bellman_expectation_Q(P, R, pi, gamma, Q):
    n_states, n_actions, _ = P.shape
    Q_new = np.zeros_like(Q)
    for s in range(n_states):
        for a in range(n_actions):
            value = 0.0
            for s_next in range(n_states):
                next_value = np.sum(pi[s_next] * Q[s_next])
                value += P[s, a, s_next] * (R[s, a, s_next] + gamma * next_value)
            Q_new[s, a] = value
    return Q_new

Q = np.zeros((n_states, n_actions))
for i in range(10):
    Q = bellman_expectation_Q(P, R, pi, gamma, Q)
    print(f"Iteration {i}:\nQ =\n{Q}\n")

Iteration 0:
Q =
[[0.8 2. ]
 [2.  4. ]
 [0.  0. ]]

Iteration 1:
Q =
[[2.168 3.8  ]
 [2.9   4.   ]
 [0.    0.   ]]

Iteration 2:
Q =
[[3.47048 4.61   ]
 [3.305   4.     ]
 [0.      0.     ]]

Iteration 3:
Q =
[[4.3038728 4.9745   ]
 [3.48725   4.       ]
 [0.        0.       ]]

Iteration 4:
Q =
[[4.76791921 5.138525  ]
 [3.5692625  4.        ]
 [0.         0.        ]]

Iteration 5:
Q =
[[5.00878716 5.21233625]
 [3.60616812 4.        ]
 [0.         0.        ]]

Iteration 6:
Q =
[[5.12871469 5.24555131]
 [3.62277566 4.        ]
 [0.         0.        ]]

Iteration 7:
Q =
[[5.18683538 5.26049809]
 [3.63024905 4.        ]
 [0.         0.        ]]

Iteration 8:
Q =
[[5.21448488 5.26722414]
 [3.63361207 4.        ]
 [0.         0.        ]]

Iteration 9:
Q =
[[5.22746542 5.27025086]
 [3.63512543 4.        ]
 [0.         0.        ]]



### Interpretation of Action-Value Function Results

The action-value function \(Q^\pi(s,a)\) shows the expected return of taking action \(a\) in state \(s\) and then following policy \(\pi\).

Key observations:

- \(Q(s,1)\) is consistently higher than \(Q(s,0)\) in states where action 1 leads to higher rewards or better transitions.
- The values converge over iterations, similar to the state-value function.
- Comparing \(Q(s,0)\) and \(Q(s,1)\) reveals which action is preferred under the policy. For example, in state 0, action 1 yields a higher value, aligning with the reward structure.

This illustrates how the Bellman expectation equation evaluates the quality of actions under a fixed policy.


## 4. Bellman Optimality Equation for V*

In [10]:
def bellman_optimality_V(P, R, gamma, V):
    n_states, n_actions, _ = P.shape
    V_new = np.zeros_like(V)
    for s in range(n_states):
        action_values = []
        for a in range(n_actions):
            value = 0.0
            for s_next in range(n_states):
                value += P[s, a, s_next] * (R[s, a, s_next] + gamma * V[s_next])
            action_values.append(value)
        V_new[s] = max(action_values)
    return V_new

V_star = np.zeros(n_states)
for i in range(10):
    V_star = bellman_optimality_V(P, R, gamma, V_star)
    print(f"Iteration {i}: V* = {V_star}")

Iteration 0: V* = [2. 4. 0.]
Iteration 1: V* = [5.6 4.  0. ]
Iteration 2: V* = [5.6 4.  0. ]
Iteration 3: V* = [5.6 4.  0. ]
Iteration 4: V* = [5.6 4.  0. ]
Iteration 5: V* = [5.6 4.  0. ]
Iteration 6: V* = [5.6 4.  0. ]
Iteration 7: V* = [5.6 4.  0. ]
Iteration 8: V* = [5.6 4.  0. ]
Iteration 9: V* = [5.6 4.  0. ]


### Interpretation of Optimal State-Value Function Results

The optimal value function \(V^*\) represents the maximum achievable expected return from each state under any policy.

Important points:

- \(V^*\) is greater than or equal to \(V^\pi\) for all states because it assumes optimal action selection.
- The values converge quickly because the optimal actions produce consistent transitions.
- State 0 has the highest optimal value due to its high-reward transitions.
- State 2 remains at zero because no action yields positive reward or transitions to beneficial states.

This demonstrates how value iteration identifies the best possible long-term returns for each state.


## 5. Bellman Optimality Equation for Q*

In [11]:
def bellman_optimality_Q(P, R, gamma, Q):
    n_states, n_actions, _ = P.shape
    Q_new = np.zeros_like(Q)
    for s in range(n_states):
        for a in range(n_actions):
            value = 0.0
            for s_next in range(n_states):
                max_next = np.max(Q[s_next])
                value += P[s, a, s_next] * (R[s, a, s_next] + gamma * max_next)
            Q_new[s, a] = value
    return Q_new

Q_star = np.zeros((n_states, n_actions))
for i in range(10):
    Q_star = bellman_optimality_Q(P, R, gamma, Q_star)
    print(f"Iteration {i}:\nQ* =\n{Q_star}\n")

Iteration 0:
Q* =
[[0.8 2. ]
 [2.  4. ]
 [0.  0. ]]

Iteration 1:
Q* =
[[2.96 5.6 ]
 [3.8  4.  ]
 [0.   0.  ]]

Iteration 2:
Q* =
[[5.552 5.6  ]
 [3.8   4.   ]
 [0.    0.   ]]

Iteration 3:
Q* =
[[5.552 5.6  ]
 [3.8   4.   ]
 [0.    0.   ]]

Iteration 4:
Q* =
[[5.552 5.6  ]
 [3.8   4.   ]
 [0.    0.   ]]

Iteration 5:
Q* =
[[5.552 5.6  ]
 [3.8   4.   ]
 [0.    0.   ]]

Iteration 6:
Q* =
[[5.552 5.6  ]
 [3.8   4.   ]
 [0.    0.   ]]

Iteration 7:
Q* =
[[5.552 5.6  ]
 [3.8   4.   ]
 [0.    0.   ]]

Iteration 8:
Q* =
[[5.552 5.6  ]
 [3.8   4.   ]
 [0.    0.   ]]

Iteration 9:
Q* =
[[5.552 5.6  ]
 [3.8   4.   ]
 [0.    0.   ]]



### Interpretation of Optimal Action-Value Function Results

The optimal action-value function \(Q^*(s,a)\) shows the best possible expected return when taking action \(a\) in state \(s\) and then acting optimally.

Key insights:

- The greedy action for each state is the one with the highest \(Q^*(s,a)\).
- In state 0, action 1 is optimal because it leads to the highest long-term reward.
- In state 1, action 1 is optimal due to its transition to a high-reward next state.
- State 2 remains zero because it is an absorbing state with no reward.

These results illustrate how the Bellman optimality equation identifies the optimal policy.


## 6. Second MDP Example: Effect of Changing Transitions, Rewards, and Policy

To demonstrate how value functions change when the environment or policy changes, this section introduces a second MDP with:

- Different transition probabilities
- Different reward structure
- A different policy \( \pi_2 \)

This comparison highlights how Bellman equations respond to structural differences in the MDP.


In [12]:
P2 = np.array([
    [
        [0.6, 0.4, 0.0],   # state 0, action 0
        [0.1, 0.9, 0.0]    # state 0, action 1
    ],
    [
        [0.0, 0.3, 0.7],   # state 1, action 0
        [0.0, 0.0, 1.0]    # state 1, action 1
    ],
    [
        [0.0, 0.0, 1.0],   # state 2, action 0
        [0.0, 0.0, 1.0]    # state 2, action 1
    ]
])

R2 = np.array([
    [
        [0.5, 1.0, 0.0],   # state 0, action 0
        [0.0, 3.0, 0.0]    # state 0, action 1
    ],
    [
        [0.0, 0.5, 2.0],   # state 1, action 0
        [0.0, 0.0, 5.0]    # state 1, action 1
    ],
    [
        [0.0, 0.0, 0.0],   # state 2, action 0
        [0.0, 0.0, 0.0]    # state 2, action 1
    ]
])

# New policy for second MDP
pi2 = np.array([
    [0.2, 0.8],   # state 0: prefer action 1
    [0.5, 0.5],   # state 1: mix actions
    [1.0, 0.0]    # state 2: always action 0
])

print("Second MDP loaded.")


Second MDP loaded.


### 6.1 Bellman Expectation for V under Policy π₂


In [13]:
V2 = np.zeros(n_states)
print("Initial V2:", V2)

for i in range(10):
    V2 = bellman_expectation_V(P2, R2, pi2, gamma, V2)
    print(f"Iteration {i}: V2 = {V2}")


Initial V2: [0. 0. 0.]
Iteration 0: V2 = [2.3   3.275 0.   ]
Iteration 1: V2 = [5.072    3.717125 0.      ]
Iteration 2: V2 = [5.88929    3.77681187 0.        ]
Iteration 3: V2 = [6.07937675 3.7848696  0.        ]
Iteration 4: V2 = [6.11939393 3.7859574  0.        ]
Iteration 5: V2 = [6.12738023 3.78610425 0.        ]
Iteration 6: V2 = [6.1289235  3.78612407 0.        ]
Iteration 7: V2 = [6.12921556 3.78612675 0.        ]
Iteration 8: V2 = [6.12927006 3.78612711 0.        ]
Iteration 9: V2 = [6.12928013 3.78612716 0.        ]


### Interpretation of V₂ Results

Several differences appear compared to the first MDP:

- State 0 has higher values because action 1 now yields a reward of 3.0 and transitions strongly to state 1.
- State 1 has higher values due to the 5.0 reward available from action 1.
- State 2 remains at zero because it is still an absorbing state with no reward.

This demonstrates how changes in reward structure and policy directly influence the expected long-term returns.


### 6.2 Bellman Expectation for Q under Policy π₂


In [14]:
Q2 = np.zeros((n_states, n_actions))
print("Initial Q2:\n", Q2)

for i in range(10):
    Q2 = bellman_expectation_Q(P2, R2, pi2, gamma, Q2)
    print(f"Iteration {i}:\nQ2 =\n{Q2}\n")


Initial Q2:
 [[0. 0.]
 [0. 0.]
 [0. 0.]]
Iteration 0:
Q2 =
[[0.7  2.7 ]
 [1.55 5.  ]
 [0.   0.  ]]

Iteration 1:
Q2 =
[[3.121   5.55975]
 [2.43425 5.     ]
 [0.      0.     ]]

Iteration 2:
Q2 =
[[4.777045   6.16735125]
 [2.55362375 5.        ]
 [0.         0.        ]]

Iteration 3:
Q2 =
[[5.23986887 6.28925372]
 [2.56973921 5.        ]
 [0.         0.        ]]

Iteration 4:
Q2 =
[[5.3454165  6.31288829]
 [2.57191479 5.        ]
 [0.         0.        ]]

Iteration 5:
Q2 =
[[5.36741738 6.31737094]
 [2.5722085  5.        ]
 [0.         0.        ]]

Iteration 6:
Q2 =
[[5.37178286 6.31820866]
 [2.57224815 5.        ]
 [0.         0.        ]]

Iteration 7:
Q2 =
[[5.37262336 6.31836361]
 [2.5722535  5.        ]
 [0.         0.        ]]

Iteration 8:
Q2 =
[[5.37278203 6.31839207]
 [2.57225422 5.        ]
 [0.         0.        ]]

Iteration 9:
Q2 =
[[5.37281159 6.31839727]
 [2.57225432 5.        ]
 [0.         0.        ]]



### Interpretation of Q₂ Results

Key observations:

- Action 1 in state 0 becomes significantly more valuable due to higher reward and better transitions.
- In state 1, action 1 dominates because it leads directly to a high reward of 5.0.
- State 2 remains zero because both actions lead to a zero-reward absorbing state.

These results show how the Bellman expectation equation evaluates the quality of actions under a different policy and environment.


## 6.3 Comparison Between First and Second MDP

This second MDP example highlights several important concepts:

- **Rewards matter:** Increasing the reward for an action directly increases both V and Q values.
- **Transitions matter:** Higher probability of transitioning to a rewarding state increases long-term value.
- **Policies matter:** A policy that favors high-reward actions produces higher value estimates.
- **Absorbing states remain low-value:** State 2 stays at zero in both MDPs because no reward is available.

This comparison reinforces how Bellman equations capture the structure of the environment and the behavior of the policy.


## 7. Summary of Key Concepts

This notebook demonstrates the core ideas behind Bellman equations in reinforcement learning:

- The **Bellman expectation equation** evaluates a policy by combining immediate rewards with discounted future values.
- The **Bellman optimality equation** identifies the best possible long-term returns by maximizing over actions.
- **State-value functions** summarize the quality of states.
- **Action-value functions** summarize the quality of actions.
- Iterative updates converge to fixed points that represent either the value of a policy or the optimal value function.
- Differences in transitions, rewards, and policies directly affect the computed values.

Together, these results show how dynamic programming methods evaluate and optimize decision-making in Markov Decision Processes.


## 8. AI Use Disclosure
AI assistance was used to help structure the notebook and generate example implementations. All code and explanations were reviewed and understood before inclusion.